In [ ]:
# EXAONE 엑사
# [system]
# 시스템지시자
# [user]
# 사용자 질문/지시
# [assistnat]

# Transformers의 토크나이져에 내장된 apply_chat_template() --> 딕셔너리 리스트구조(message)를 기반 토큰처리

In [ ]:
from transformers import AutoTokenizer
model_id = "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# 대화 목록 구조설계
prompt = '하늘이 파란이유를 간단하게 한문장으로 설명해줘'
messages = [
    {"role": "system", 
     "content": "너는 친절하고 명확하게 답변하는 인공지능 페르소나 입니다."},
    {"role": "user", "content": prompt}
]
# apply_chat_template 적용( tokenize=False  텍스트 형태의 문자열 결과를 먼저 검증)
# add_generaton_prompt=True 답변영역인 [Asisstant]를 맨뒤에 자동으로 붙여 생성을 시작하게 함
formatted_prompt =  tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
formatted_prompt

In [ ]:
import torch
from transformers import AutoModelForCausalLM
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16
    device_map = "auto"
    low_cpu_mem_usage = True
else:
    torch_dtype = torch.float32
    device_map = "cpu"
    low_cpu_mem_usage = True
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map=device_map,
    low_cpu_mem_usage=low_cpu_mem_usage,
    trust_remote_code=True
)


In [ ]:
# 디코딩 매개변수 : 텍스트 생성을 통제
# temperature : 확률분포 낮을수록 높은 확률의 단어만 선택, 결정론적이고 일관된 글이 생성
# 높은수록 다양하고 창의적인 어휘가 발휘
# top-p(Nucleus Sampling): 누적확률 분포가 P이하인 상위 단어풀에서만 선택
# Repetition Penalty : 이미 등장한 단어가 불필요하게 반복 생성되는 것을 방지

# transformer 계열에는 패딩토큰(pad_token_id) 문장종료토큰 (eos_token_id) 이 같은 토큰 eos_token으로 지정
# apply_chat_template 통하면 input_ids 배열만 추출해서 generator에 보내면 어디까지가 문장이고 어디까지가 패딩
# 인지 유추할수 없다 경고발생
# 해결방법 : apply_chat_template의 결과가 단일 torch tensor가 아니라 input_ids, attention_maks가 구분되는
# 딕셔너리로 생성하도록
def generate_text_with_params(messages_list: list, temperature: float, top_p: float, repetition_penalty: float):
    """
    apply_chat_template을 직접 사용하여 설정된 디코딩 파라미터로 EXAONE 모델을 통해 답변을 생성합니다.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # tokenizer.apply_chat_template()을 활용하여 다이렉트로 PyTorch Tensor(tokenize=True)를 추출합니다.
    input_ids = tokenizer.apply_chat_template(
        messages_list,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True  # 딕셔너리반환(attention_mask포함)
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **input_ids,  # return_dict=True  or  return_dict가 없으면 input_ids로 입력
            max_new_tokens=150,
            do_sample=True if temperature > 0.0 else False,
            temperature=temperature if temperature > 0.0 else None,
            top_p=top_p if temperature > 0.0 else None,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 입력으로 들어갔던 프롬프트(input_ids) 영역을 제외하고 모델이 생성한 순수 답변 텍스트 영역만 디코딩합니다.
    print(input_ids)
    input_length = input_ids['input_ids'].shape[1]  # return_dict=True or True가아니면 input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()



In [ ]:
# 일관성(Greedy-like) vs 창의성(Sampling) 비교
# 일관성 모드 : 정답이 정해져 있는 논리적 태스트에 유용(temperature = 0.1, top_p=0.9,re... = 1.1)
# 창의성 모드 : 이야기새성, 마케팅문구 작성에 유리(temperature=0.9, top_p = 0.95, re.... = 1.2)

# 일관성 중심 생성
logical_messages = [
    {'role':'system','content':'너는 백과사전 수준의 논리적으로 정답만을 말하는 페르소나입니다.'},
    {'role':'user','content':'하늘이 파랗게 보이는 과학적 원리는 무엇인가요?'}
]

result_logical = generate_text_with_params(logical_messages,temperature=0.1, top_p=0.9,repetition_penalty=1.1)
print(result_logical)

In [ ]:
# 창의성 중심 생성
logical_messages = [
    {'role':'system','content':'너는 감성적이고 트랜드한 카피라이터야'},
    {'role':'user','content':'새로출시한 손목시계의 홍보용 sns 문구를 20대 타겟으로 작성해줘 '}
]

result_logical = generate_text_with_params(logical_messages,temperature=0.9, top_p=0.95,repetition_penalty=1.2)
print(result_logical)

In [ ]:
# 한국어 문서 생성 요약
# 시스템프롬프트 조절을 통해서 한줄 핵심요약, 3줄 개조식 요약, 단락형 요약등 다양한 출력 스타일 제어
# Few Shot Prompting을 활용해서 금융/법률 등 전문도메인에 맞는 트화된 용약 양식과 톤앤매너를 유도
# 길이가 긴 텍스트 입력시 발생할수 있는 컨텍스트관리 및 오류대응

In [ ]:
news_article = """
[서울=IT뉴스] 인공지능(AI) 반도체 시장의 선두 주자인 엔비디아가 차세대 AI 칩 '블랙웰(Blackwell)'의 본격적인 양산에 돌입했다고 발표했습니다.
엔비디아의 젠슨 황 최고경영자(CEO)는 22일 열린 연례 개발자 컨퍼런스에서 "블랙웰 칩은 이전 세대 제품인 '호퍼(Hopper)'보다 AI 연산 속도가 최대 30배 빠르고, 전력 소비량은 최대 25배 효율적"이라고 강조했습니다.
업계 관계자들에 따르면 블랙웰 칩은 구글, 메타, 마이크로소프트 등 글로벌 빅테크 기업들의 데이터센터에 가장 먼저 배치될 예정입니다.
최근 환경 보호와 고성능 컴퓨팅이 상충하는 데이터센터 업계에서 전력 효율 극대화는 필수 과제로 꼽혀왔으며, 엔비디아의 이번 블랙웰 발표가 친환경 AI 인프라 구축의 분수령이 될 것이라는 전망이 지배적입니다.
한편 국내 메모리 반도체 제조사들 역시 블랙웰에 탑재될 5세대 고대역폭메모리(HBM3E) 공급을 본격화하면서 수혜를 입을 것으로 보입니다.
전문가들은 "HBM 메모리 부족 현상이 지속되는 가운데, 국내 기업들의 고부가가치 D램 제품 수출이 역대 최대치를 경신할 가능성이 있다"고 내다봤습니다.
"""
print(f'뉴스기사 글자수 : {len(news_article)}')

In [ ]:
# 디코딩 매개변수 : 텍스트 생성을 통제
# temperature : 확률분포 낮을수록 높은 확률의 단어만 선택, 결정론적이고 일관된 글이 생성
# 높은수록 다양하고 창의적인 어휘가 발휘
# top-p(Nucleus Sampling): 누적확률 분포가 P이하인 상위 단어풀에서만 선택
# Repetition Penalty : 이미 등장한 단어가 불필요하게 반복 생성되는 것을 방지

# transformer 계열에는 패딩토큰(pad_token_id) 문장종료토큰 (eos_token_id) 이 같은 토큰 eos_token으로 지정
# apply_chat_template 통하면 input_ids 배열만 추출해서 generator에 보내면 어디까지가 문장이고 어디까지가 패딩
# 인지 유추할수 없다 경고발생
# 해결방법 : apply_chat_template의 결과가 단일 torch tensor가 아니라 input_ids, attention_maks가 구분되는
# 딕셔너리로 생성하도록
def generate_text_summary(systemp_prompt:str ,article:str,max_tokens:int=150):
    """
    시스템 지시문과 뉴스 기사 본문을 입력받아 요약문을 생성
    """
    messages = [
        {'role':'system','content':systemp_prompt},
        {'role':'user','content':f'아래 뉴스기스를 읽고 요약해줘:\n{article}'}
    ]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # tokenizer.apply_chat_template()을 활용하여 다이렉트로 PyTorch Tensor(tokenize=True)를 추출합니다.
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True  # 딕셔너리반환(attention_mask포함)
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **input_ids,  # return_dict=True  or  return_dict가 없으면 input_ids로 입력
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.3,            
            repetition_penalty=1.1, # 불필요한 반복 억제
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 입력으로 들어갔던 프롬프트(input_ids) 영역을 제외하고 모델이 생성한 순수 답변 텍스트 영역만 디코딩합니다.    
    input_length = input_ids['input_ids'].shape[1]  # return_dict=True or True가아니면 input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()



In [ ]:
one_line_system = '''
너는 뉴스 기사 요약 전문가야
주어진 뉴스 기사를 읽고, 군더더기 없는 명확한 문장 **딱 한줄(한 문장)** 로 핵심 주제를 요약해줘.
추가적인 인사말이나 사족 없이 즉시 요약문만 출력해야 해
'''
one_line_summary = generate_text_summary(one_line_system,news_article,max_tokens=60)
print(one_line_summary)

In [ ]:
# 스타일 2 : 3줄 개조식(Bullet Point)요약
bullet_system = '''
너는 문서 핵심 정보 요약기야
주어진 텍스트의 중요 포인트 3가지를 정리해야 해.
반드시 아래의 양식을 정확히 지켜줘

[요약 양식]
- {첫 번째 핵심 사건이나 발표}
- {두 번째 주요 특징이나 수치}
- {세 번째 시장 반응 또는 업계 영향}
'''
one_line_summary = generate_text_summary(bullet_system,news_article,max_tokens=150)
print(one_line_summary)


In [ ]:
# 스타일3 : 분석적 보고서 요약(핵심단어 해시태그 포함)
hashtag_system = '''
너는 빅데이터 뉴스 기사 분석기야.
기사를 읽고 2~3문장으로 구성된 자연스러운 단락 형태의 요약문을 먼저 작성해줘.
그리고 요약문 작성이 끝나면, 다음 줄에 해당 기사를 대표하는 해시태그를 3개 작성해줘.
예: #키워드1 #키워드2 #키워드3
'''
one_line_summary = generate_text_summary(hashtag_system,news_article,max_tokens=150)
print(one_line_summary)

In [ ]:
# 1. 실습용 타겟 금융/투자 뉴스 텍스트
target_finance_article = """
바이오 제약 벤처인 '메디코어'는 오늘 임상 3상 시험 결과가 기준치 미달로 나와 최종 승인이 보류되었다고 공시했습니다.
이번 3상 실패 발표 직후 메디코어의 주가는 개장하자마자 하한가로 직행하며 시장에 큰 충격을 주었습니다.
메디코어 측은 "부작용은 관찰되지 않았으나 주평가 지표의 유의성을 확보하지 못했다"고 설명하며, 향후 투여 용량을 재조정하여 추가 분석 임상을 진행하겠다는 계획을 덧붙였습니다.
하지만 투자 애널리스트들은 이번 보류 결정으로 인해 최소 2년간의 추가 개발 기간과 추가 자금 수백억 원의 조달이 불가피할 것으로 보고 있습니다.
최근 바이오 업계 전반의 자금 경색이 지속되는 가운데 메디코어의 신용 등급 강등 우려가 나오자 관련 바이오 섹터 지수 전체가 동반 하락하고 있는 상황입니다.
"""

# 2. Few-shot 데이터 구성
# 시스템 프롬프트에 원하는 포맷 규칙을 입력하고,
# User/Assistant 롤플레이를 통해 가이드 예시(Shot)를 1개 포함하여 모델에게 전달합니다.
def run_few_shot_summary(target_article: str):
    system_prompt = """
너는 투자 전문 애널리스트들의 보고서를 요약하는 분석 비서야.
전달받는 기업 뉴스나 보고서에 대해 반드시 다음 3가지 분석적 포맷을 엄격하게 준수하여 결과를 도출해야 해:
1. [이슈의 요지] : 무슨 일이 일어났는지 핵심 팩트를 한 줄 요약
2. [리스크 요인] : 기업이나 시장에 미치는 부정적인 잠재 위협
3. [전망 및 결론] : 향후 타임라인이나 투자 관점에서의 결론
"""

    # Few-shot 예시 1개 구성
    few_shot_user_example = """
아래 뉴스 기사를 읽고 요약해줘:
전자부품 제조업체 '테크텍'의 2분기 영업이익이 전년 동기 대비 40% 감소한 것으로 드러났습니다. 주요 거래처인 스마트폰 제조사의 수요 둔화와 원자재 가격 상승이 원인입니다. 테크텍 측은 하반기 차량용 반도체 부품 신사업 진출을 통해 돌파구를 마련할 예정이나, 아직 신사업 매출 비중은 3% 미만에 불과하여 실적 회복까지는 시간이 걸릴 것이라는 것이 증권가의 지배적인 평가입니다.
"""
    
    few_shot_assistant_example = """
1. [이슈의 요지] : 스마트폰 수요 둔화 및 원자재비 상승으로 테크텍의 2분기 영업이익이 전년 대비 40% 급감함.
2. [리스크 요인] : 원자재 수입 의존도가 높아 가격 변동에 취약하며 실적 부진 장기화 리스크 존재함.
3. [전망 및 결론] : 하반기 신사업(차량용 반도체) 진출이 예정되어 있으나 매출 비중이 극히 낮아 단기 실적 회복은 불투명함.
"""

    # 실무 메시지 리스트 구축 (Few-shot 1쌍 추가)
    messages = [
        {"role": "system", "content": system_prompt},
        # Few-shot 예시 1
        {"role": "user", "content": few_shot_user_example},
        {"role": "assistant", "content": few_shot_assistant_example.strip()},
        # 실제 입력 데이터
        {"role": "user", "content": f"아래 뉴스 기사를 읽고 요약해줘:\n{target_article}"}
    ]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.2, # 형식을 철저히 따르기 위해 온도를 더 낮춥니다.
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    # BatchEncoding 객체는 딕셔너리이므로 inputs["input_ids"]로 접근합니다.
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


In [ ]:
print( run_few_shot_summary(target_article=target_finance_article) )

In [ ]:
# VRAM 리소스 정리
import gc

# 사용하지 않는 가비지 수집 및 PyTorch CUDA 캐시 정리
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU Memory cache cleared successfully.")
else:
    print("CPU environment. No GPU cache clearance needed.")

* EXAONE 3.0 인스트럭션 튜닝 모델을 제어하여 영-한 및 한-영 번역 모듈을 설계합니다.
* 구조화되지 않은 텍스트(예: 비즈니스 이메일)로부터 필요한 정보들만 스키마에 따라 온전히 파싱하여 JSON 포맷으로 강제 출력하는 프롬프트 제어 기술을 습득합니다.
* 모델의 최종 텍스트 출력을 Python `json` 라이브러리로 직접 파싱하여 데이터 정제 파이프라인을 완성하고 오류 발생 유무를 검증합니다.

In [ ]:
def translate_text(text:str, source_lang:str, target_lang:str):
    systemp_prompt = f'''너는 원문의 의미와 뉘앙스를 자연스럽고 정교하게 전달하는 전문 번역사야. 
    주언진 문장을 {source_lang}에서 {target_lang}으로 번역해줘
    번역외의 다른 추가적인 추론이나 기타 불필요한 문장은 생성하지 말것
    '''
    messages = [
        {'role':'system','content':systemp_prompt},
        {'role':'user','content':f'번역할 문장:\n{text}'}
    ]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.2, # 형식을 철저히 따르기 위해 온도를 더 낮춥니다.
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    # BatchEncoding 객체는 딕셔너리이므로 inputs["input_ids"]로 접근합니다.
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    

In [ ]:
#영한 번역 테스트
en_sentence = "US President Donald Trump has raised the prospect of speaking to Taiwan’s President Lai Ching-te, in what would be an unprecedented move for a US leader and a major departure from diplomatic norms."
kor_translation = translate_text(en_sentence,'영어','한국어')
print(kor_translation)

In [ ]:
#영한 번역 테스트
kor_sentence = "미국 대통령 도널드 트럼프가 대만 대통령 라이 칭티와 대화할 가능성을 제기했는데, 이는 미국 지도자로서 전례 없는 조치이며 외교 규범에서 큰 이탈이다."
eng_translation = translate_text(kor_sentence,'한국어','영어')
print(eng_translation)

In [ ]:
# 실습용 비정형 영문 비즈니스 이메일 텍스트
email_text = """
Subject: Urgent: Kick-off Meeting for Project Alpha Next Week
From: Sarah Jenkins (sarah.j@alpha-tech.com)
To: Michael Chen (michael.c@alpha-tech.com)
Date: May 22, 2026

Hi Michael,

Hope you are doing well. 
We need to schedule our project kick-off meeting next week. I am proposing next Tuesday (May 26th) at 10:00 AM EST. Let me know if this works for you. 
Please make sure to invite the DevOps team, especially David. Also, could you prepare the initial system design diagram before the session? We will need to present it to the stakeholders.

Best regards,
Sarah Jenkins
"""
print("이메일 본문 길이:", len(email_text), "자")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import json

def extract_info_to_json(email_content: str):
    """
    이메일 내용을 해석하여 미리 약속한 JSON 형식으로 추출합니다.
    """
    # 시스템 프롬프트에 출력하고자 하는 JSON 키 구조와 양식을 지시합니다.
    system_prompt = """
너는 비정형 텍스트 문서에서 중요 정보만 요약 추출하는 데이터 정제 엔진이야. 
제공된 텍스트에서 정보를 해독하여 아래의 JSON 구조에 완벽하게 일치하게 데이터를 채워서 오직 JSON 데이터만 출력해줘.
추가 설명이나 인사말, 사족, 백틱(```) ,** **,```json,이모티콘 등을 절대로 섞지 말고 오로지 순수한 JSON 문자열 자체만 리턴해야만 해.

{
  "sender": "보낸 사람의 이름과 이메일 주소",
  "recipient": "받는 사람의 이름과 이메일 주소",
  "meeting_proposal": {
    "date": "제안된 회의 일자 (YYYY-MM-DD 포맷)",
    "time": "제안된 시간"
  },
  "attendees": ["회의 참석 요구 인물 목록 (문자열 리스트)"],
  "tasks_to_do": ["받는 사람이 회의 전 수행해야 할 과업 목록 (문자열 리스트)"]
}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"아래 이메일에서 정보를 추출하여 JSON으로 리턴해줘:\n{email_content}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False, # JSON 구조 파싱 정밀도를 극대화하기 위해 Sampling을 배제합니다.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# 정보 추출 함수 실행
raw_json_output = extract_info_to_json(email_text)
print("--- 모델의 원본 텍스트 출력 ---")
print(raw_json_output)
print("\n" + "="*60 + "\n")

* 지문(Context) 내에 사실 정보가 포함되지 않았을 때 환각 답변을 원천적으로 제한하는 네거티브 조건 설정법을 익힙니다.
* 복잡한 추론이나 산출 과정이 요구될 때 **Few-shot Prompting**과 **Chain-of-Thought(CoT)** 기법을 결합하여 논리적 완성도를 올립니다.
* 시스템 프롬프트(System Prompt) 상의 페르소나 제어로 사용자의 목적에 맞게 답변 톤앤매너를 변경합니다.

In [ ]:
mrc_context = """
인천국제공항공사는 오는 10월부터 인천공항 제2여객터미널(T2)의 확장 구역을 본격 개장한다고 발표했습니다.
이번 확장 사업은 2017년부터 총 4조 8,000억 원의 공사비가 투입된 4단계 건설 사업의 핵심으로, 연간 여객 수용 능력이 기존 7,700만 명에서 1억 600만 명으로 약 37% 확대될 예정입니다.
확장된 공간에는 최첨단 스마트 방역 및 지능형 출입국 시스템이 적용되었으며, 승객들의 이동 효율성을 극대화하기 위해 셀프 백드롭(수하물 위탁) 기기가 대폭 증설되었습니다.
다만, 최근 항공 유가 급등과 지상 조업사들의 인력 수급 불균형 문제로 인해 공항 면세 구역 내 신규 브랜드 매장 중 약 20%는 개장일인 10월에 즉시 입점하지 못하고 12월에 순차적으로 오픈할 계획인 것으로 알려졌습니다.
"""
print(f"MRC 지문 글자 수: {len(mrc_context)} 자")

###  할루시네이션(환각) 통제를 위한 강력한 네거티브 조건 설정
지문에 명시되지 않은 정보를 모델이 유추하거나 지어내어 그럴싸하게 거짓말하는 할루시네이션을 최소화하기 위해 시스템 프롬프트 규정을 타이트하게 조절합니다.
- `return_dict=True`를 전달하여 어텐션 마스크 경고를 예방합니다.
- `inputs["input_ids"].shape[1]` 슬라이싱을 이용해 입력 프롬프트를 배제한 결과만 추출합니다.

In [ ]:
def run_mrc_qa(context: str, question: str):
    """
    주어진 지문에만 기반하여 질문에 대답하도록 제어하는 MRC 함수입니다.
    """
    system_prompt = """
너는 주어진 본문에만 철저하게 근거하여 정답을 추출하는 사실 확인기야.
반드시 제공하는 [본문]의 텍스트 영역 안에서만 정답을 도출해야 해.
만약 질문의 정답이 [본문]에 직접 나와 있지 않거나 확인 불가능한 내용이라면, 절대 억지로 추론하거나 상상하여 지어내지 말고 반드시 "지문에 제공되지 않은 정보이므로 답변할 수 없습니다."라고 단호하게 리턴해야 해.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"[본문]\n{context}\n\n[질문]\n{question}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False, # 사실 관계 검증이므로 Greedy Search로 무작위성을 없앱니다.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# 테스트 1: 지문에 정보가 존재하는 경우의 질문
q1 = "제2여객터미널 확장 사업을 위해 투입된 총 공사비는 얼마인가요?"
ans1 = run_mrc_qa(mrc_context, q1)
print(f"질문: {q1}")
print(f"답변: {ans1}")
print("\n" + "="*60 + "\n")

# 테스트 2: 지문에 정보가 존재하지 않는 경우의 질문 (환각 억제 작동 체크)
q2 = "확장 개장일 당일에 모든 브랜드 면세점들이 100% 문을 열지 못하는 지상 조업사 이외의 구체적인 원인은 무엇인가요?"
# 지문에는 '항공 유가 급등과 지상 조업사들의 인력 수급 불균형 문제'라고 나와 있으나 지상 조업사 외의 구체적인 회사 사정이나 개별 브랜드 사유는 없습니다.
ans2 = run_mrc_qa(mrc_context, q2)
print(f"질문: {q2}")
print(f"답변: {ans2}")

### Few-shot Prompting과 Chain-of-Thought (CoT) QA
질문의 난이도가 높거나, 단계적인 논리 전개 및 산출 구조(예: 수학 연산, 조건부 매핑 등)가 필요할 때는 예제 답변(Few-shot)과 함께 추론 단계를 기록하는 CoT를 유도하는 것이 효과적입니다.

In [ ]:
# 복잡한 산술 및 추론형 지문 데이터 정의
logic_context = """
철수는 냉장고에서 사과 5개와 오렌지 3개를 꺼내 식탁 위에 두었습니다.
동생인 영희가 방에서 나오더니 식탁 위에 있던 과일 중 오렌지 2개를 가지고 방으로 다시 들어갔습니다.
그 후 철수는 식탁 위에 사과 2개를 더 올려두었고, 오렌지 1개는 깎아 먹었습니다.
"""

def run_few_shot_cot_qa(context: str, question: str):
    """
    Few-shot 예제에 Chain-of-Thought 논리를 가미하여 복잡한 과일 수량 계산을 제어합니다.
    """
    system_prompt = """
너는 문장을 단계별로 쪼개어 수량을 추론하고 수학적 계산 과정을 소상히 밝혀 정답을 제시하는 논리 추론 도우미야.
반드시 제공받은 본문의 시간 순서대로 과일 개수의 가감 과정을 서술한 뒤 최종 과일의 총 개수를 도출해야 해.
"""

    # Few-shot 예제 1쌍 준비
    few_shot_user_example = """
[본문]
민수는 바구니에 바나나 10개를 넣어 두었습니다. 미영이가 바구니에서 바나나 4개를 꺼내 먹었고, 민수가 바나나 3개를 바구니에 다시 넣어 두었습니다.

[질문]
바구니에 남은 최종 바나나 개수는 총 몇 개입니까?
"""
    
    few_shot_assistant_example = """
[추론 단계]
1. 최초 민수가 바구니에 넣어 둔 바나나는 10개입니다.
2. 미영이가 바나나 4개를 꺼내 먹었으므로 남은 개수는 10 - 4 = 6개입니다.
3. 민수가 바나나 3개를 다시 넣었으므로 남은 개수는 6 + 3 = 9개입니다.

[최종 답변] 바구니에 남은 최종 바나나의 개수는 9개입니다.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        # Few-shot Shot 1
        {"role": "user", "content": few_shot_user_example},
        {"role": "assistant", "content": few_shot_assistant_example.strip()},
        # 실제 태스크 입력
        {"role": "user", "content": f"[본문]\n{context}\n\n[질문]\n{question}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False, # 추론 태스크에서는 샘플링을 끄는 것이 좋습니다.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

target_q = "현재 식탁 위에 올려져 있는 과일(사과와 오렌지)의 총 개수는 몇 개입니까?"
logic_ans = run_few_shot_cot_qa(logic_context, target_q)
print(f"질문: {target_q}")
print(logic_ans)

### 질의응답 모델 페르소나 제어 실습
동일한 지문과 동일한 질문일지라도 시스템 프롬프트에 규정된 페르소나(친절한 유치원 선생님 vs 차갑고 기계적인 인공지능 분석 기계)에 따라 출력 톤앤매너가 완벽하게 다르게 반응하는지 검증합니다.

In [ ]:
def run_persona_qa(context: str, question: str, persona_prompt: str):
    """
    전달받은 페르소나 지시어에 맞춰 톤앤매너가 변형된 정답을 도출합니다.
    """
    messages = [
        {"role": "system", "content": persona_prompt},
        {"role": "user", "content": f"[본문]\n{context}\n\n[질문]\n{question}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7, # 톤앤매너의 어휘 다양성을 위해 약간의 온도를 제공합니다.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# 공통 지문 및 질문
target_context = "제 4단계 건설 사업에 총 4조 8,000억 원의 공사비가 투입되었습니다."
target_question = "4단계 사업에 들어간 총 공사비가 얼마인가요?"

# 페르소나 A: 친절한 유치원 선생님
persona_a = "너는 7살 어린이들에게 친절하게 대답해 주는 유치원 선생님이야. 구어체와 이모티콘을 적극 사용하여 아주 다정하고 알기 쉽게 대답해줘."
ans_a = run_persona_qa(target_context, target_question, persona_a)
print("[페르소나 A: 유치원 선생님]")
print(ans_a)
print("\n" + "="*60 + "\n")

# 페르소나 B: 차갑고 냉철한 인공지능 기계
persona_b = "너는 감정이 없는 매우 차갑고 냉철한 기계 장치 시스템이야. 정중한 경어체나 사족을 일절 배제하고 단어로만 명료하고 직설적이게 팩트만 대답해줘."
ans_b = run_persona_qa(target_context, target_question, persona_b)
print("[페르소나 B: 인공지능 기계]")
print(ans_b)

사용자의 자연어 설명(요구사항)을 구체적인 알고리즘/프로그래밍 소스코드로 변환하는 **Text-to-Code** 프롬프팅을 학습합니다.
* 논리 오류나 구문 오류가 있는 코드를 스스로 분석하여 버그를 찾고 해결 방안을 친절하게 설명하는 **코드 디버거** 기능을 구현합니다.
* 기작성된 코드를 분석해 기능별로 설명하고, 정합성 높은 한국어 주석과 독스트링(Docstring)을 자동으로 삽입하는 **코드 설명 및 문서화기**를 만들어 봅니다.

In [ ]:
def generate_code(instruction: str):
    """
    사용자의 자연어 요구사항을 바탕으로 Python 코드를 생성합니다.
    """
    system_prompt = """
너는 사용자의 자연어 설명과 요구사항을 바탕으로 오직 작동 가능하고 완성도 높은 파이썬(Python) 코드만을 자동 생성하는 실력 있는 코딩 전문가야.
설명은 배제하고, 작성된 코드는 반드시 ```python ... ``` 형태의 마크다운 코드 블록으로 포맷을 감싸서 제시해줘.
코드 안에는 적절한 핵심 주석을 한국어로 달아줘.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"[요구사항]\n{instruction}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,  # 코딩과 같이 명확한 구문과 출력이 정해진 작업에서는 Greedy search를 권장합니다.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# 실습 1: 데이터프레임 조작 및 시각화 코드 생성
requirement = """
pandas 데이터프레임을 생성해줘. 데이터프레임은 '도시'(서울, 부산, 인천, 대구, 광주)와 '인구수(만명)'(950, 330, 300, 240, 145)를 담아야 해.
그 뒤 matplotlib를 사용하여 도시별 인구수를 그리는 막대 그래프(Bar Chart)를 시각화하는 파이썬 코드를 작성해줘.
한글 깨짐 현상을 처리하기 위한 코드 주석과 설정도 포함해줘.
"""

generated_result = generate_code(requirement)
print("[생성된 소스코드]")
print(generated_result)